In [ ]:
from pathlib import Path
from astroquery.gaia import Gaia
from astropy.table import Table
import pandas as pd
import numpy as np
import sys
import os

from tqdm.auto import tqdm

In [25]:
# Cargamos datos del ADQL
gaia_query_path = Path("services", "queries", "gaia_sdss_random_subset.adql")
gaia_query_template = gaia_query_path.read_text(encoding="utf-8")

# índices random
gaia_query = gaia_query_template.format(start_index=30000, end_index=50000)

gaia_query_job = Gaia.launch_job_async(gaia_query)
gaia_job_results = gaia_query_job.get_results() or Table()

display(gaia_job_results)

# Convertimos a panda/dataset
gaia_df = gaia_job_results.to_pandas()
# gaia_df.columns = [c.lower() for c in gaia_df.columns]

# Preprocesamos datos de Gaia

ispec_params = gaia_df.rename(columns={
    "teff_gspphot": "teff",
    "logg_gspphot": "logg",
    "mh_gspphot": "mh"
}).copy()

# Quitar filas no completas
ispec_params = ispec_params.dropna(
    subset=["source_id", "teff", "logg", "mh"]
).reset_index(drop=True)

print("Filas con parámetros completos:", len(ispec_params))

# Por el momento seteamos alpha a 0 constante
ispec_params["alpha"] = 0

############ Recomendación de CHATY tengo que mirar si es cierto
ispec_params = ispec_params[
    (ispec_params["teff"] >= 3500) &
    (ispec_params["teff"] <= 8000) &
    (ispec_params["logg"] >= 0.0) &
    (ispec_params["logg"] <= 5.5) &
    (ispec_params["mh"] >= -2.5) &
    (ispec_params["mh"] <= 0.5)
].reset_index(drop=True)

display(ispec_params[["source_id", "teff", "logg", "mh", "alpha"]].head())

# Guardamos datos en csv
params_output_dir = Path("data", "processed")
params_output_dir.mkdir(parents=True, exist_ok=True)

params_output_path = params_output_dir / "gaia_params_for_ispec.csv"
ispec_params.to_csv(params_output_path, index=False)

[2026-06-02 23:12:53,959] [INFO] [core:launch_job_async:481]: Query finished.


INFO: Query finished. [astroquery.utils.tap.core]


source_id,ra,dec,phot_bp_mean_mag,phot_rp_mean_mag,rvs_spec_sig_to_noise,astrometric_excess_noise,bp_rp,teff_gspphot,logg_gspphot,mh_gspphot,vbroad,alphafe_gspspec,original_ext_source_id,angular_distance
,deg,deg,mag,mag,,mas,mag,K,log(cm.s**-2),dex,km / s,dex,,arcsec
int64,float64,float64,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,int64,float32
1012764493967581312,138.44081572620522,48.707029416929785,15.907301,14.464811,--,0.0,1.4424896,4371.032,4.4257,-0.2672,--,--,1237657772316950577,0.05089582
1589762907457297280,228.47946990984158,50.97969443927994,16.189617,15.30423,--,0.0,0.8853874,5425.385,4.59,-0.6584,--,--,1237659149387563016,0.067105286
1958150911092920960,334.4817569110048,42.173753956590375,17.30849,16.226406,--,0.1261946,1.0820847,5051.276,4.3534,-1.5257,--,--,1237672765505274972,0.048141275
2229421732702064128,335.4800540362377,70.79800730216758,15.858689,14.688266,--,0.026899735,1.1704235,6906.9883,4.1846,-0.628,--,--,1237663231224512722,0.025595
97386635385758976,28.10748363988089,21.85421052784815,14.414324,13.328199,--,0.2053607,1.0861244,5289.0303,4.5875,-0.4661,--,--,1237679543510630469,0.07904631
2537260489983199360,14.631352760200368,0.5547298902553643,12.545439,11.724998,--,0.4418147,0.8204403,5613.0776,4.0685,-0.4826,--,0.25,1237663784740651043,0.061714645
3244145410356301184,54.69880922901643,-7.044003537448336,17.932938,17.155907,--,0.045636553,0.77703094,5761.0117,4.6812,-1.5383,--,--,1237649961920299065,0.10979005
150784005274051072,64.78867111871637,26.514776377879635,18.106668,16.98221,--,0.0623774,1.1244583,4808.3057,4.8496,-2.249,--,--,1237660554912465548,0.06373736


Filas con parámetros completos: 237


,source_id,teff,logg,mh,alpha
0,1012764493967581312,4371.032227,4.4257,-0.2672,0
1,1589762907457297280,5425.384766,4.5900,-0.6584,0
2,1958150911092920960,5051.275879,4.3534,-1.5257,0
3,2229421732702064128,6906.988281,4.1846,-0.6280,0
4,97386635385758976,5289.030273,4.5875,-0.4661,0


In [26]:
# Cargamos ispec
ISPEC_PATH = "/Users/carlasequero/iSpec"

if ISPEC_PATH not in sys.path:
    sys.path.insert(0, ISPEC_PATH)

import ispec

print("iSpec:", ispec.__file__)
print("SPECTRUM activo:", ispec.is_spectrum_support_enabled())

# Parametros a configurar segun lo que creamos conveniente
WAVE_MIN = 450.0
WAVE_MAX = 900.0
WAVE_STEP = 0.02

# SDSS es 2000, la he puesto un poco mas alta
R_HIGH = 25000

# Muy importante: para iSpec/SPECTRUM debe ser float64
wave_high = np.arange(
    WAVE_MIN,
    WAVE_MAX + WAVE_STEP / 2,
    WAVE_STEP,
    dtype=np.float64
)

# Rutas predefinidas para encontrar variables que ispec usará
atomic_linelist_file = "/Users/carlasequero/iSpec/input/linelists/transitions/VALD.300_1100nm/atomic_lines.tsv"
solar_abundances_file = "/Users/carlasequero/iSpec/input/abundances/Grevesse.2007/stdatom.dat"
isotope_file = "/Users/carlasequero/iSpec/input/isotopes/SPECTRUM.lst"
model_atmospheres_dir = "/Users/carlasequero/iSpec/input/atmospheres/MARCS.GES/"


atomic_linelist = ispec.read_atomic_linelist(
    atomic_linelist_file,
    wave_base=WAVE_MIN,
    wave_top=WAVE_MAX
)

# Cargamos los recursos que necesita ispec
isotopes = ispec.read_isotope_data(isotope_file)
solar_abundances = ispec.read_solar_abundances(solar_abundances_file)
modeled_layers_pack = ispec.load_modeled_layers_pack(model_atmospheres_dir)

iSpec: /Users/carlasequero/iSpec/ispec/__init__.py
SPECTRUM activo: True


In [ ]:
# Dependiendo de los modelos atmosfericos que tiene cargados, no podrá generar todos los datos, asi que filtramos los que si
# QUIZAS PODEMOS QUITARLO Y YA QUE SE COMPRUEBA LUEGO DE NUEVO

def is_valid_ispec_target(row):
    target = {
        "teff": float(row["teff"]),
        "logg": float(row["logg"]),
        "MH": float(row["mh"]),
        "alpha": float(row["alpha"])
    }
    
    return ispec.valid_atmosphere_target(
        modeled_layers_pack,
        target
    )

ispec_params["valid_atmosphere"] = ispec_params.apply(
    is_valid_ispec_target,
    axis=1
)

print(ispec_params["valid_atmosphere"].value_counts())

ispec_params_valid = ispec_params[
    ispec_params["valid_atmosphere"]
].reset_index(drop=True)

print("Estrellas válidas para iSpec:", len(ispec_params_valid))

display(
    ispec_params_valid[
        ["source_id", "teff", "logg", "mh", "alpha", "valid_atmosphere"]
    ].head()
)

valid_atmosphere
True    228
Name: count, dtype: int64
Estrellas válidas para iSpec: 228


,source_id,teff,logg,mh,alpha,valid_atmosphere
0,1012764493967581312,4371.032227,4.4257,-0.2672,0,True
1,1589762907457297280,5425.384766,4.5900,-0.6584,0,True
2,1958150911092920960,5051.275879,4.3534,-1.5257,0,True
3,2229421732702064128,6906.988281,4.1846,-0.6280,0,True
4,97386635385758976,5289.030273,4.5875,-0.4661,0,True


In [28]:
def generate_ispec_spectrum_from_params(teff, logg, mh, alpha, wave, atomic_linelist, isotopes, solar_abundances, modeled_layers_pack, R=2500):  
    wave = np.asarray(wave, dtype=np.float64)
    teff = float(teff)
    logg = float(logg)
    mh = float(mh)
    alpha = float(alpha)

    target = {
        "teff": teff,
        "logg": logg,
        "MH": mh,
        "alpha": alpha
    }

    if not ispec.valid_atmosphere_target(modeled_layers_pack, target):
        raise ValueError(f"Parámetros fuera de grid atmosférica: {target}")

    atmosphere_layers = ispec.interpolate_atmosphere_layers(
        modeled_layers_pack,
        target,
        code="spectrum"
    )

    # HAY QUE MIRAR COMO DEFINIR ESTO
    microturbulence = 1.0
    macroturbulence = 2.0
    vsini = 2.0
    limb_darkening_coeff = 0.6

    flux = ispec.generate_spectrum(
        wave,
        atmosphere_layers,
        teff,
        logg,
        mh,
        alpha,
        atomic_linelist,
        isotopes,
        solar_abundances,
        fixed_abundances = None,
        microturbulence_vel = microturbulence,
        macroturbulence = macroturbulence,
        vsini = vsini,
        limb_darkening_coeff = limb_darkening_coeff,
        R = R,
        regions = None,
        verbose = 0,
        code = "spectrum"
    )

    flux = np.asarray(flux, dtype=np.float32)

    if np.all(flux == 0):
        raise ValueError("El espectro salió todo a cero")

    return flux

In [29]:
def generate_spectra_in_batches(
    params_df,
    wave_high,
    atomic_linelist,
    isotopes,
    solar_abundances,
    modeled_layers_pack,
    output_dir,
    batch_size=50,
    R_high=2500
):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    np.save(
        output_dir / "wave_high_nm.npy",
        np.asarray(wave_high, dtype=np.float32)
    )

    params_df = params_df.copy().reset_index(drop=False).rename(
        columns={"index": "orig_index"}
    )

    failed_rows = []
    worked_rows = []

    n_total = len(params_df)
    n_batches = int(np.ceil(n_total / batch_size))

    print(f"Total de estrellas a procesar: {n_total}")
    print(f"Número de batches: {n_batches}")

    for batch_id in range(n_batches):
        start = batch_id * batch_size
        end = min((batch_id + 1) * batch_size, n_total)

        batch_df = params_df.iloc[start:end].copy().reset_index(drop=True)

        batch_fluxes = []
        batch_metadata = []

        print(f"\nProcesando batch {batch_id + 1}/{n_batches} ({start}:{end})")

        for _, row in tqdm(
            list(batch_df.iterrows()),
            total=len(batch_df),
            desc=f"Batch {batch_id + 1}/{n_batches}"
        ):
            try:
                teff = float(row["teff"])
                logg = float(row["logg"])
                mh = float(row["mh"])
                alpha = float(row["alpha"])

                flux = generate_ispec_spectrum_from_params(
                    teff=teff,
                    logg=logg,
                    mh=mh,
                    alpha=alpha,
                    wave=wave_high,
                    atomic_linelist=atomic_linelist,
                    isotopes=isotopes,
                    solar_abundances=solar_abundances,
                    modeled_layers_pack=modeled_layers_pack,
                    R=R_high
                )

                flux = np.asarray(flux, dtype=np.float32)

                batch_fluxes.append(flux)

                meta = {
                    "source_id": row["source_id"] if "source_id" in row else np.nan,
                    "orig_index": int(row["orig_index"]),
                    "row_in_batch": len(batch_fluxes) - 1,
                    "batch_id": batch_id,
                    "teff": teff,
                    "logg": logg,
                    "mh": mh,
                    "alpha": alpha,
                    "normalized": False
                }

                batch_metadata.append(meta)
                worked_rows.append(meta)

            except Exception as e:
                failed_rows.append({
                    "source_id": row["source_id"] if "source_id" in row else np.nan,
                    "orig_index": int(row["orig_index"]),
                    "batch_id": batch_id,
                    "teff": row.get("teff", np.nan),
                    "logg": row.get("logg", np.nan),
                    "mh": row.get("mh", np.nan),
                    "alpha": row.get("alpha", np.nan),
                    "error": str(e)
                })

        if len(batch_fluxes) > 0:
            batch_fluxes = np.vstack(batch_fluxes).astype(np.float32)
            batch_metadata_df = pd.DataFrame(batch_metadata)

            spectra_path = output_dir / f"spectra_high_batch_{batch_id:04d}.npy"
            metadata_path = output_dir / f"metadata_batch_{batch_id:04d}.csv"

            np.save(spectra_path, batch_fluxes)
            batch_metadata_df.to_csv(metadata_path, index=False)

            print(f"Guardado: {spectra_path}")

    worked_df = pd.DataFrame(worked_rows)
    failed_df = pd.DataFrame(failed_rows)

    worked_df.to_csv(output_dir / "worked_spectra.csv", index=False)
    failed_df.to_csv(output_dir / "failed_rows.csv", index=False)

    print("\n=== RESUMEN FINAL ===")
    print("Espectros generados:", len(worked_df))
    print("Fallidos:", len(failed_df))
    print("Espectros generados:", output_dir / "worked_spectra.csv")
    print("Fallidos:", output_dir / "failed_rows.csv")

    return worked_df, failed_df

In [ ]:
output_path = Path("data", "synthetic_high_res_450_900_raw")

output_path.mkdir(parents=True, exist_ok=True)

manifest_full_df, failed_full_df = generate_spectra_in_batches(
    params_df=ispec_params_valid,
    wave_high=wave_high,
    atomic_linelist=atomic_linelist,
    isotopes=isotopes,
    solar_abundances=solar_abundances,
    modeled_layers_pack=modeled_layers_pack,
    output_dir=output_path,
    batch_size=50,
    R_high=R_HIGH
)

print("Generados:", len(manifest_full_df))
print("Fallidos:", len(failed_full_df))

if len(failed_full_df) > 0:
    display(failed_full_df.head())
    display(failed_full_df["error"].value_counts().head(10))

Total de estrellas a procesar: 228
Número de batches: 5

Procesando batch 1/5 (0:50)


Batch 1/5:   0%|          | 0/50 [00:00<?, ?it/s]

[2026-06-03 20:41:33,300] [INFO] [utils:_init_num_threads:160]: NumExpr defaulting to 8 threads.
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (
[2026-06-03 20:41:52,475] [INFO] [utils:_init_num_threads:160]: NumExpr defaulting to 8 threads.
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: Use

Guardado: data/synthetic_high_res_450_900_raw/spectra_high_batch_0000.npy

Procesando batch 2/5 (50:100)


Batch 2/5:   0%|          | 0/50 [00:00<?, ?it/s]

[2026-06-03 20:53:58,589] [INFO] [utils:_init_num_threads:160]: NumExpr defaulting to 8 threads.
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (
[2026-06-03 20:54:12,421] [INFO] [utils:_init_num_threads:160]: NumExpr defaulting to 8 threads.
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: Use